In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json

import utilities.functions as functions
from utilities.functions import (
    retidos,
    calcula_viabilidade,
    analisar_retencao,
)

pd.set_option('display.float_format', '{:,.2f}'.format)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
import os
import os
os.getcwd()
import os
import pandas as pd
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 


In [ ]:
df = pd.read_parquet(BASE_PATH / "gold" / "df_publico.parquet")
id_outlier=df[
    df['outlier_iqr'] & 
    df['outlier_zscore'] & 
    df['outlier_mad']]['customer_id']
len(id_outlier)

In [ ]:

publico_janeiro_dezembro = pd.read_parquet(BASE_PATH / "gold" / "df_clientes.parquet")


In [ ]:
df_outlier=publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(id_outlier)]
publico_janeiro_dezembro['is_outlier'] = publico_janeiro_dezembro['customer_id'].isin(id_outlier).astype(int)

In [ ]:
id_both_monht=publico_janeiro_dezembro[publico_janeiro_dezembro['order_created_month']==12]['customer_id'].unique()
publico_janeiro_dezembro = publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(id_both_monht)].reset_index(drop=True)

In [ ]:
publico_janeiro_dezembro.head()

In [ ]:
dois_pedidos=publico_janeiro_dezembro[publico_janeiro_dezembro['num_pedidos_hist']==3]
dois_pedidos.head()

dois_pedidos=publico_janeiro_dezembro[publico_janeiro_dezembro['pedidos_sum']=='2']['customer_id'].unique()
amostra_aleatoria =publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(dois_pedidos)]
df_stats_mes = amostra_aleatoria.groupby(['order_created_month', 'pedidos_sum']).agg(
    total_clientes=('customer_id', 'nunique')
).round(2)
matriz_migracao(amostra_aleatoria,mes_0=12,mes_1=1,group_by_extra='pedidos_sum')

Calculando retencao considerando dois ou + pedidos

In [ ]:
df_1,clientes_retidos_1=retidos(publico_janeiro_dezembro, mes0=12, mes1=1,pedidos=1)
print(clientes_retidos_1)

In [ ]:
retencao_pedido_1=analisar_retencao(df_1)
retencao_pedido_1

Calculando retencao considerando tres ou + pedidos

In [ ]:
df_2,clientes_retidos_2=retidos(publico_janeiro_dezembro, mes0=12, mes1=1,pedidos=2)
print(clientes_retidos_2)

In [ ]:
retencao_pedido_2=analisar_retencao(df_2)
retencao_pedido_2

Viabilidade 

In [ ]:
resultados = calcula_viabilidade(
    publico_janeiro_dezembro,
    mes_campanha=12,
    mes_seguinte=1,
    coupon_value=10.0,   
    margin_rate=0.12     
)

resultados


Calculando retencao separadamente para outliers

In [ ]:
publico_janeiro_dezembro.head()

In [ ]:
df_outlier=publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(id_outlier)]
publico_janeiro_dezembro['is_outlier'] = publico_janeiro_dezembro['customer_id'].isin(id_outlier).astype(int)


In [ ]:
publico_janeiro_dezembro.to_parquet(BASE_PATH / "gold" / "publico_janeiro_dezembro.parquet", index=False)

In [ ]:
df_out,clientes_retidos_out=retidos(df_outlier, mes0=12, mes1=1,pedidos=2)
print(clientes_retidos_out)
retencao_pedido_out=analisar_retencao(df_out)
#retencao_pedido_out

In [ ]:
resultados_out = calcula_viabilidade(
    df_outlier,
    mes_campanha=12,
    mes_seguinte=1,
    coupon_value=10.0,   
    margin_rate=0.12     
)

resultados_out
